# 03 · BioBERT: symptom extractor + evidential concept classifier
Base: `dmis-lab/biobert-base-cased-v1.2`. Colab T4 is enough (≈15–30 min per run).

* **Extractor:** stage 1 generic SYMPTOM spans on MACCROBAT → stage 2 concept-specific spans on StigmaSymp-W.
* **Criteria classifier:** multi-label, per-label binary Dirichlet (evidential) head; negated mentions are negatives.

In [ ]:
# Colab setup: clone the repo and install training extras
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("RareCare"):
    subprocess.run(["git", "clone", "https://github.com/nush1729/RareCare.git"], check=True)
    os.chdir("RareCare")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print("cwd:", os.getcwd())

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
SEED = 13

In [ ]:
# Stage 1 (optional but recommended): MACCROBAT generic symptom spans
!python data/scripts/convert_maccrobat.py --src data/raw/maccrobat --out data/maccrobat
!python -m training.train_text extractor --generic --data data/maccrobat --out checkpoints/stage1 --epochs 3 --fp16

In [ ]:
# Stage 2: concept-specific extractor
!python -m training.train_text extractor --init checkpoints/stage1 --data data/stigmasymp_w --out checkpoints/biobert-extractor --epochs 5 --fp16

In [ ]:
!python -m training.train_text criteria --data data/stigmasymp_w --out checkpoints/biobert-criteria --epochs 5

In [ ]:
import json

for ck in ["checkpoints/biobert-extractor", "checkpoints/biobert-criteria"]:
    print(ck, json.load(open(f"{ck}/eval.json")))

## Multi-seed runs (report mean ± std, never a single seed)

In [ ]:
for seed in [13, 17, 23]:
    !python -m training.train_text criteria --data data/stigmasymp_w --out checkpoints/biobert-criteria-s{seed} --seed {seed} --epochs 5

## Publish to the Hugging Face Hub (then set the checkpoint ids in `configs/trained.yaml`)

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
# from transformers import AutoModelForTokenClassification, AutoTokenizer
# AutoModelForTokenClassification.from_pretrained("checkpoints/biobert-extractor").push_to_hub("nush1729/rarecare-biobert-extractor", private=True)
# AutoTokenizer.from_pretrained("checkpoints/biobert-extractor").push_to_hub("nush1729/rarecare-biobert-extractor", private=True)